# Project HOLLYWOOD — Graph Theory Pipeline

**Pipeline:** Raw Features → Laplacian Score (correlation kNN) → UMAP → HDBSCAN → Ollama Naming

No RobustScaler — correlation distance is scale-invariant, and feature magnitude
carries real signal for genome data. Laplacian Score weights features by how
smoothly they vary across the manifold (graph Laplacian on a kNN graph).

## 1 — Configuration

In [ ]:
# ── Sweep Toggles ────────────────────────────────────────────────────────────
# Enable these to run parameter grid searches before the production pipeline.
# Results are stored in DataFrames for analysis; production params are NOT changed.
RUN_LAPLACIAN_SWEEP = False
RUN_UMAP_SWEEP      = False
RUN_HDBSCAN_SWEEP   = False

# ── Data Options ─────────────────────────────────────────────────────────────
# INCLUDE_GENRE_FEATURES: Add one-hot genre columns (~23 binary columns).
INCLUDE_GENRE_FEATURES  = True

# INCLUDE_DECADE_FEATURE: Add raw decade values (1920, 1930, ..., 2020).
#   Raw values — correlation distance and Laplacian handle scale natively.
INCLUDE_DECADE_FEATURE  = True

# ── Laplacian Score Settings ─────────────────────────────────────────────────
# LAPLACIAN_K_NEIGHBORS: Neighbors for the kNN graph. Higher k captures broader
#   structure but smooths local detail. Should be in the 10-30 range.
LAPLACIAN_K_NEIGHBORS   = 20

# LAPLACIAN_METRIC: Distance metric for the kNN graph.
#   'correlation'  — Pearson correlation, scale-invariant, captures co-variation
#   'cosine'       — angle-based, ignores magnitude, good for sparse features
#   'euclidean'    — magnitude-sensitive, dominated by scale differences
LAPLACIAN_METRIC        = 'correlation'

# ── UMAP Settings ────────────────────────────────────────────────────────────
# UMAP_N_COMPONENTS: Dimensionality of the clustering embedding. 10-50 typical.
UMAP_N_COMPONENTS       = 30

# UMAP_N_NEIGHBORS: Balances local vs global structure. Low=fine detail, high=broad.
UMAP_N_NEIGHBORS        = 15

# UMAP_MIN_DIST: Minimum distance in embedding. 0.0 = tightest packing for clustering.
UMAP_MIN_DIST           = 0.0

# UMAP_METRIC: Distance metric on Laplacian-weighted features.
#   'correlation' — scale-invariant, ideal since we skip RobustScaler.
UMAP_METRIC             = 'correlation'

# Visualisation UMAP (separate — always 3D for scatter plots)
UMAP_VIS_N_COMPONENTS   = 3
UMAP_VIS_N_NEIGHBORS    = 15
UMAP_VIS_MIN_DIST       = 0.1

# ── HDBSCAN Settings ────────────────────────────────────────────────────────
# HDBSCAN_MIN_CLUSTER_SIZE: Smallest group recognised as a cluster.
HDBSCAN_MIN_CLUSTER_SIZE = 60

# HDBSCAN_MIN_SAMPLES: Core point threshold. Higher = stricter, more outliers.
HDBSCAN_MIN_SAMPLES      = 15

# HDBSCAN_EPSILON: Distance threshold for merging micro-clusters. 0.0 = pure HDBSCAN.
HDBSCAN_EPSILON          = 0.1

# HDBSCAN_SELECTION_METHOD: 'eom' = larger persistent clusters, 'leaf' = finest-grained.
HDBSCAN_SELECTION_METHOD = 'eom'

# ── LLM Naming (Ollama) ─────────────────────────────────────────────────────
OLLAMA_MODEL         = 'llama3.2:3b'
OLLAMA_BASE_URL      = 'http://localhost:11434'

# ── Paths ────────────────────────────────────────────────────────────────────
GENOME_SCORES_PATH = 'feature_data_longform.csv'
GENOME_TAGS_PATH   = 'feature_taxonomy.csv'
OMDB_DATA_DIR      = 'omdb_data'
RESULTS_DIR        = 'results'

print('Configuration loaded.')
print(f'  Laplacian: k={LAPLACIAN_K_NEIGHBORS}, metric={LAPLACIAN_METRIC}')
print(f'  UMAP:      {UMAP_N_COMPONENTS}D, n_neighbors={UMAP_N_NEIGHBORS}, min_dist={UMAP_MIN_DIST}, metric={UMAP_METRIC}')
print(f'  HDBSCAN:   min_size={HDBSCAN_MIN_CLUSTER_SIZE}, min_samples={HDBSCAN_MIN_SAMPLES}, eps={HDBSCAN_EPSILON}, {HDBSCAN_SELECTION_METHOD}')
print(f'  Ollama:    {OLLAMA_MODEL} @ {OLLAMA_BASE_URL}')
print(f'  Sweeps:    Laplacian={RUN_LAPLACIAN_SWEEP}, UMAP={RUN_UMAP_SWEEP}, HDBSCAN={RUN_HDBSCAN_SWEEP}')

## 2 — Imports

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.neighbors import kneighbors_graph
from sklearn.impute import SimpleImputer
from skfeature.utility.construct_W import construct_W
from skfeature.function.similarity_based import lap_score
import umap
import hdbscan

print('Imports ready')

## 3 — Load Genome Data

In [ ]:
features_long = pd.read_csv(GENOME_SCORES_PATH)
taxonomy = pd.read_csv(GENOME_TAGS_PATH)

if 'tagId' in features_long.columns:
    features_long = features_long.rename(columns={
        'tagId': 'feature_id', 'movieId': 'imdb_id', 'relevance': 'trigger'
    })
if 'tagId' in taxonomy.columns:
    taxonomy = taxonomy.rename(columns={'tagId': 'feature_id', 'tag': 'feature'})

features_long = features_long.merge(taxonomy, on='feature_id', how='left')

feature_matrix = features_long.pivot_table(
    index='imdb_id', columns='feature', values='trigger', aggfunc='first'
).fillna(0)

print(f'Genome matrix: {feature_matrix.shape[0]:,} movies x {feature_matrix.shape[1]} features')

## 3b — Load OMDB Data + Extract Metadata

Loads OMDB movie data and extracts metadata (year, rating, language, decade).
Produces `jordan_df`, `title_lookup`, and `tt_codes`.

In [ ]:
from pathlib import Path
import json as _json_loader

# ── Load OMDB movie data ────────────────────────────────────────────────────
omdb_dir = Path(OMDB_DATA_DIR)
movies_file = omdb_dir / 'omdb_movies.json'

if movies_file.exists():
    with open(movies_file) as f:
        movies_data = _json_loader.load(f)
    print(f'OMDB data loaded: {len(movies_data):,} movies')
else:
    movies_data = {}
    print(f'⚠ OMDB data not found at {movies_file} — genre/decade/naming will be limited')

# ── Build dataset IMDb ID list (aligned with feature matrix) ─────────────────
dataset_imdb_ids = [str(mid) for mid in feature_matrix.index.tolist()]
tt_codes = dataset_imdb_ids

# ── Extract Jordan's metadata from OMDB responses ───────────────────────────
jordan_rows = []
for imdb_id in dataset_imdb_ids:
    movie = movies_data.get(imdb_id, movies_data.get(str(imdb_id), {}))
    if not movie:
        jordan_rows.append({
            'imdb_id': imdb_id,
            'year': 'Unknown', 'rating': 'Unknown', 'language': 'Unknown',
            'content_category': 'Unknown', 'decade': 0,
            'label_lang_rating': 'Unknown - Unknown',
            'label_lang_rating_decade': 'Unknown - Unknown - Unknown Decade'
        })
        continue

    year_raw = movie.get('Year', 'Unknown')
    rating   = movie.get('Rated', 'Unknown')
    if rating == 'N/A':
        rating = 'Unknown'
    lang = movie.get('Language', 'Unknown').split(',')[0].strip()

    try:
        year_clean = int(str(year_raw)[:4])
    except (ValueError, TypeError):
        year_clean = 0

    rating_map = {
        'G': 'Family', 'PG': 'Family', 'TV-G': 'Family', 'TV-Y': 'Family', 'TV-Y7': 'Family',
        'PG-13': 'Teen', '12A': 'Teen', 'TV-14': 'Teen', 'TV-PG': 'Teen', 'TV-Y7-FV': 'Teen',
        'R': 'Mature', 'NC-17': 'Mature', 'TV-MA': 'Mature', 'X': 'Mature',
        'Not Rated': 'Unknown', 'Unrated': 'Unknown', 'N/A': 'Unknown', 'Unknown': 'Unknown',
        'Approved': 'Unknown', 'Passed': 'Unknown', 'GP': 'Unknown', 'M': 'Unknown', 'M/PG': 'Unknown',
    }
    content_category = rating_map.get(rating, 'Unknown')
    decade = (year_clean // 10) * 10

    label_lr = f'{lang} - {content_category}'
    label_lrd = f'{lang} - {content_category} - {decade}s' if decade > 0 else f'{lang} - {content_category} - Unknown Decade'

    jordan_rows.append({
        'imdb_id': imdb_id,
        'year': year_raw, 'rating': rating, 'language': lang,
        'content_category': content_category, 'decade': decade,
        'label_lang_rating': label_lr,
        'label_lang_rating_decade': label_lrd
    })

jordan_df = pd.DataFrame(jordan_rows)

# Save CSV for reuse
jordan_csv = omdb_dir / 'jordan_metadata.csv'
if omdb_dir.exists():
    jordan_df.to_csv(jordan_csv, index=False)

# Build title lookup for LLM naming prompts
title_lookup = {}
for imdb_id, movie in movies_data.items():
    title = movie.get('Title', '')
    if title:
        title_lookup[imdb_id] = title

print(f'Jordan metadata: {jordan_df.shape[0]:,} rows')
print(f'  Rating categories: {jordan_df["content_category"].value_counts().to_dict()}')
print(f'  Decade range: {sorted(jordan_df[jordan_df["decade"] > 0]["decade"].unique())}')
print(f'  Title lookup: {len(title_lookup):,} movies with titles')

## 4 — Optional Features (Genre + Decade)

In [ ]:
import json as _json

# ── Genre one-hot ────────────────────────────────────────────────────────────
if INCLUDE_GENRE_FEATURES:
    genre_path = Path(OMDB_DATA_DIR) / 'omdb_movies.json'
    if genre_path.exists():
        with open(genre_path) as f:
            omdb_raw = _json.load(f)

        genre_rows = []
        for imdb_id, entry in omdb_raw.items():
            genres = entry.get('Genre', '')
            if genres and genres != 'N/A':
                for g in genres.split(','):
                    genre_rows.append({'imdb_id': imdb_id, 'genre': g.strip()})

        if genre_rows:
            genre_df = pd.DataFrame(genre_rows)
            genre_df['_val'] = 1
            genre_onehot = genre_df.pivot_table(
                index='imdb_id', columns='genre', values='_val', aggfunc='max'
            ).fillna(0)
            genre_onehot.columns = [f'genre_{c}' for c in genre_onehot.columns]

            existing = [c for c in feature_matrix.columns if c.startswith('genre_')]
            if existing:
                feature_matrix = feature_matrix.drop(columns=existing)
            feature_matrix = feature_matrix.join(genre_onehot, how='left').fillna(0)
            print(f'Genre features added: {len(genre_onehot.columns)} columns')
    else:
        print(f'Genre file not found — skipping')
else:
    existing = [c for c in feature_matrix.columns if c.startswith('genre_')]
    if existing:
        feature_matrix = feature_matrix.drop(columns=existing)
    print('Genre features: SKIPPED')

# ── Decade ───────────────────────────────────────────────────────────────────
existing_decade = [c for c in feature_matrix.columns if c == 'meta_decade']
if existing_decade:
    feature_matrix = feature_matrix.drop(columns=existing_decade)

if INCLUDE_DECADE_FEATURE:
    jordan_path = Path(OMDB_DATA_DIR) / 'jordan_metadata.csv'
    if jordan_path.exists():
        jordan_df = pd.read_csv(jordan_path)
        jf = jordan_df.set_index('imdb_id')
        decade_col = jf[['decade']].copy()
        decade_col.columns = ['meta_decade']
        decade_col.index.name = 'imdb_id'
        feature_matrix = feature_matrix.join(decade_col, how='left').fillna(0)
        print(f'Decade feature added (raw values — correlation distance handles scale natively)')
    else:
        print(f'Jordan metadata not found — skipping decade')
else:
    print('Decade feature: SKIPPED')

print(f'\nFeature matrix: {feature_matrix.shape[0]:,} x {feature_matrix.shape[1]}')

## 5 — Pre-processing: Clean

No RobustScaler — correlation distance (used by both Laplacian and UMAP) is
scale-invariant, and feature magnitude carries real signal for genome data.
We only remove constants and impute missing values.

In [ ]:
binary_cols    = [c for c in feature_matrix.columns if feature_matrix[c].nunique() == 2]
continuous_cols = [c for c in feature_matrix.columns if c not in binary_cols]

print(f'Feature breakdown: {len(continuous_cols)} continuous, {len(binary_cols)} binary')

# Remove constant features
constant_cols = [c for c in feature_matrix.columns if feature_matrix[c].nunique() <= 1]
if constant_cols:
    print(f'Dropping {len(constant_cols)} constant features')
    feature_matrix = feature_matrix.drop(columns=constant_cols)
    binary_cols    = [c for c in binary_cols if c not in constant_cols]
    continuous_cols = [c for c in continuous_cols if c not in constant_cols]

# Impute missing values
if feature_matrix.isnull().any().any():
    imputer = SimpleImputer(strategy='median')
    feature_matrix[continuous_cols] = imputer.fit_transform(feature_matrix[continuous_cols])
    feature_matrix[binary_cols] = feature_matrix[binary_cols].fillna(0)

X = feature_matrix.values.astype(np.float32)
feature_names = feature_matrix.columns.tolist()

print(f'Full matrix: {X.shape[0]:,} x {X.shape[1]}')
print(f'Sparsity: {(X == 0).mean():.1%}')

## 6 — Laplacian Score Weighting

Measures how smoothly each feature varies across the kNN graph.
Low Laplacian score = feature preserves local manifold structure = important.
Weights are inverted scores mapped to [0, 1]. No L2 normalization — magnitude matters.

In [ ]:
def laplacian_score(X, k=20, metric='correlation'):
    n_samples, n_features = X.shape

    print(f'  Building kNN graph (k={k}, metric={metric}, n={n_samples})...')

    if metric in ('euclidean', 'cosine'):
        # construct_W supports these natively
        W = construct_W(X, metric=metric, neighbor_mode='knn', weight_mode='binary',
                        k=min(k, n_samples - 1))
    else:
        # For 'correlation' or other metrics, build via sklearn and convert
        from scipy.sparse import csc_matrix
        knn = kneighbors_graph(X, n_neighbors=min(k, n_samples - 1),
                               metric=metric, mode='connectivity', include_self=False)
        W = csc_matrix((knn + knn.T).astype(float))
        W[W > 0] = 1.0

    print(f'  Computing Laplacian scores ({n_features} features)...')
    scores = lap_score.lap_score(X, mode='raw', W=W)
    scores = np.asarray(scores).flatten()

    # Invert to weights: low score = important = high weight, mapped to [0, 1]
    valid = np.isfinite(scores) & (scores < 1e10)
    weights = np.zeros(n_features)
    if valid.any():
        min_s, max_s = scores[valid].min(), scores[valid].max()
        if max_s > min_s:
            weights[valid] = 1.0 - (scores[valid] - min_s) / (max_s - min_s)
        else:
            weights[valid] = 1.0

    return scores, weights


print('Computing Laplacian Scores...')
lap_scores, lap_weights = laplacian_score(X, k=LAPLACIAN_K_NEIGHBORS, metric=LAPLACIAN_METRIC)

# Apply weights (no L2 — magnitude differences carry real signal)
X_weighted = X * lap_weights

# Diagnostics
top_idx = np.argsort(lap_weights)[-10:][::-1]
print(f'\n  Laplacian weighting applied.')
print(f'  Weight range: [{lap_weights.min():.4f}, {lap_weights.max():.4f}]')
print(f'\n  Top 10 most important features:')
for idx in top_idx:
    print(f'    {feature_names[idx]:<45s} weight={lap_weights[idx]:.4f}')
print(f'\n  X_weighted: {X_weighted.shape}')

### 6.1 — Laplacian Score Parameter Sweep
Sweeps `k_neighbors` × `metric` combinations. Each combo rebuilds the full
Laplacian → UMAP → HDBSCAN pipeline to measure downstream clustering quality.

In [ ]:
if RUN_LAPLACIAN_SWEEP:
    from itertools import product as _product
    from sklearn.metrics import silhouette_score as _sil
    from scipy.sparse.csgraph import connected_components as _cc

    _k_list = [5, 10, 15, 20, 30, 50]
    _m_list = ['cosine', 'euclidean', 'correlation']

    lap_sweep_results = []
    _total = len(_k_list) * len(_m_list)
    print(f'Laplacian sweep: {len(_k_list)} k × {len(_m_list)} metrics = {_total} combos')
    print(f'{"k":<5} {"metric":<13} {"wt_range":<10} {"retention":<10} {"knn_comp":<10} {"clusters":<10} {"outlier%":<10} {"silhouette"}')
    print('-' * 90)

    for _k, _m in _product(_k_list, _m_list):
        try:
            _ls, _lw = laplacian_score(X, k=_k, metric=_m)

            # Feature retention: fraction of features with weight > 10% of max
            _max_w = _lw.max() if _lw.max() > 0 else 1.0
            _retained = float(np.sum(_lw > 0.1 * _max_w)) / len(_lw)

            # kNN graph connectivity
            _W = construct_W(X, metric=_m, neighbor_mode='knn', weight_mode='binary',
                             k=min(_k, X.shape[0] - 1))
            _n_comp, _ = _cc(_W, directed=False)

            _xw = X * _lw
            _red = umap.UMAP(n_components=UMAP_N_COMPONENTS, n_neighbors=UMAP_N_NEIGHBORS,
                             min_dist=UMAP_MIN_DIST, metric=UMAP_METRIC, random_state=42,
                             low_memory=False)
            _emb = _red.fit_transform(_xw)
            _cl = hdbscan.HDBSCAN(min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
                                   min_samples=HDBSCAN_MIN_SAMPLES,
                                   cluster_selection_epsilon=HDBSCAN_EPSILON,
                                   metric='euclidean',
                                   cluster_selection_method=HDBSCAN_SELECTION_METHOD)
            _lbl = _cl.fit_predict(_emb)
            _nc = len(set(_lbl)) - (1 if -1 in _lbl else 0)
            _no = int(np.sum(_lbl == -1))
            _op = _no / len(_lbl)
            _sil_val = _sil(_emb[_lbl != -1], _lbl[_lbl != -1],
                            sample_size=min(2000, int((_lbl != -1).sum())),
                            random_state=42) if _nc >= 2 else -1.0
            _wr = float(_lw.max() - _lw.min())
            lap_sweep_results.append({'k': _k, 'metric': _m, 'weight_range': round(_wr, 4),
                                      'feature_retention': round(_retained, 3),
                                      'knn_components': _n_comp,
                                      'n_clusters': _nc, 'outlier_pct': round(_op, 3),
                                      'silhouette': round(_sil_val, 3)})
            print(f'{_k:<5} {_m:<13} {_wr:<10.3f} {_retained:<10.1%} {_n_comp:<10} {_nc:<10} {_op:<10.1%} {_sil_val:.3f}')
        except Exception as _e:
            print(f'{_k:<5} {_m:<13} ERROR: {str(_e)[:50]}')

    lap_sweep_df = pd.DataFrame(lap_sweep_results)
    print('\nTop 10 by silhouette:')
    print(lap_sweep_df.sort_values('silhouette', ascending=False).head(10).to_string(index=False))

    print('\nPer-metric summary:')
    _ms = lap_sweep_df.groupby('metric').agg(
        avg_sil=('silhouette', 'mean'), best_sil=('silhouette', 'max'),
        avg_retention=('feature_retention', 'mean'),
        avg_knn_comp=('knn_components', 'mean')
    ).round(3)
    print(_ms.to_string())
else:
    print('Laplacian sweep: SKIPPED (RUN_LAPLACIAN_SWEEP = False)')

In [ ]:
try:
    lap_sweep_df
except NameError:
    print('No Laplacian sweep data — set RUN_LAPLACIAN_SWEEP = True')
else:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    _metrics = sorted(lap_sweep_df['metric'].unique())
    _mc = {'cosine': '#E74C3C', 'euclidean': '#3498DB', 'correlation': '#2ECC71'}

    fig = make_subplots(rows=2, cols=3, subplot_titles=(
        'Silhouette by Metric', 'Feature Retention by Metric', 'kNN Components by k',
        'Silhouette vs k (by metric)', 'Retention vs k (by metric)', 'Best Config per Metric'))

    for m in _metrics:
        s = lap_sweep_df[lap_sweep_df['metric'] == m]
        c = _mc.get(m, '#888')
        fig.add_trace(go.Histogram(x=s['silhouette'], name=m, marker_color=c, opacity=0.7), row=1, col=1)
        fig.add_trace(go.Histogram(x=s['feature_retention']*100, name=m, marker_color=c, opacity=0.7,
                                   showlegend=False), row=1, col=2)
        fig.add_trace(go.Scatter(x=s['k'], y=s['knn_components'], mode='markers+lines',
                                 name=m, marker=dict(size=8, color=c), showlegend=False), row=1, col=3)
        by_k = s.groupby('k').agg(sil=('silhouette', 'mean')).reset_index()
        fig.add_trace(go.Scatter(x=by_k['k'], y=by_k['sil'], mode='markers+lines',
                                 name=m, marker=dict(size=8, color=c), showlegend=False), row=2, col=1)
        by_k2 = s.groupby('k').agg(ret=('feature_retention', 'mean')).reset_index()
        fig.add_trace(go.Scatter(x=by_k2['k'], y=by_k2['ret']*100, mode='markers+lines',
                                 name=m, marker=dict(size=8, color=c), showlegend=False), row=2, col=2)

    best = lap_sweep_df.loc[lap_sweep_df.groupby('metric')['silhouette'].idxmax()]
    fig.add_trace(go.Bar(x=best['metric'], y=best['silhouette'],
                         marker_color=[_mc.get(m, '#888') for m in best['metric']],
                         text=[f'k={r.k}' for _, r in best.iterrows()],
                         textposition='auto', showlegend=False), row=2, col=3)

    fig.update_xaxes(title_text='Silhouette', row=1, col=1)
    fig.update_xaxes(title_text='Retention %', row=1, col=2)
    fig.update_xaxes(title_text='k', row=1, col=3)
    fig.update_yaxes(title_text='Components', row=1, col=3)
    fig.update_xaxes(title_text='k', row=2, col=1)
    fig.update_yaxes(title_text='Avg Silhouette', row=2, col=1)
    fig.update_xaxes(title_text='k', row=2, col=2)
    fig.update_yaxes(title_text='Avg Retention %', row=2, col=2)
    fig.update_yaxes(title_text='Best Silhouette', row=2, col=3)
    fig.update_layout(height=700, template='plotly_dark', title_text='Laplacian Sweep — Metric Comparison')
    fig.show()

## 7 — UMAP

In [ ]:
print(f'UMAP ({UMAP_METRIC}) on Laplacian-weighted features...')

reducer_cluster = umap.UMAP(
    n_components=UMAP_N_COMPONENTS, n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST, metric=UMAP_METRIC, random_state=42, low_memory=False
)
reducer_3d = umap.UMAP(
    n_components=UMAP_VIS_N_COMPONENTS, n_neighbors=UMAP_VIS_N_NEIGHBORS,
    min_dist=UMAP_VIS_MIN_DIST, metric=UMAP_METRIC, random_state=42
)

print(f'  Fitting clustering embedding ({UMAP_N_COMPONENTS}D)...')
embedding_cluster = reducer_cluster.fit_transform(X_weighted)

print(f'  Fitting visualisation embedding ({UMAP_VIS_N_COMPONENTS}D)...')
embedding_3d = reducer_3d.fit_transform(X_weighted)
embedding_2d = embedding_3d[:, :2]

print(f'\n  Clustering: {embedding_cluster.shape}')
print(f'  Visualisation: {embedding_3d.shape}')

### 7.1 — UMAP Parameter Sweep
Sweeps `n_components` × `n_neighbors` × `metric` × `min_dist` on the
Laplacian-weighted features. Measures silhouette, DBCV, trustworthiness.

In [ ]:
if RUN_UMAP_SWEEP:
    from itertools import product as _product
    from sklearn.metrics import silhouette_score as _sil, davies_bouldin_score as _db
    from sklearn.manifold import trustworthiness as _tw

    _nc_list = [10, 15, 20, 30, 50, 80]
    _nn_list = [15, 30, 50, 100]
    _um_list = ['correlation', 'cosine', 'euclidean']
    _md_list = [0.0, 0.05, 0.1]

    umap_sweep_results = []
    _total = len(_nc_list) * len(_nn_list) * len(_um_list) * len(_md_list)
    print(f'UMAP sweep: {len(_nc_list)} n_comp × {len(_nn_list)} n_neigh × {len(_um_list)} metrics × {len(_md_list)} min_dist = {_total} combos')
    print(f'{"n_comp":<8} {"n_neigh":<8} {"metric":<13} {"m_dist":<7} {"sil":<8} {"DB":<8} {"trust":<8} {"DBCV":<8} {"clust":<8} {"out%"}')
    print('-' * 95)

    for _nc, _nn, _um, _md in _product(_nc_list, _nn_list, _um_list, _md_list):
        try:
            _red = umap.UMAP(n_components=_nc, n_neighbors=_nn, min_dist=_md,
                             metric=_um, random_state=42, low_memory=False)
            _emb = _red.fit_transform(X_weighted)
            _cl = hdbscan.HDBSCAN(min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
                                   min_samples=HDBSCAN_MIN_SAMPLES,
                                   cluster_selection_epsilon=HDBSCAN_EPSILON,
                                   metric='euclidean',
                                   cluster_selection_method=HDBSCAN_SELECTION_METHOD,
                                   gen_min_span_tree=True)
            _lbl = _cl.fit_predict(_emb)
            _n_c = len(set(_lbl)) - (1 if -1 in _lbl else 0)
            _n_o = int(np.sum(_lbl == -1))
            _op = _n_o / len(_lbl)

            if _n_c >= 2:
                _v = _lbl != -1
                _s = _sil(_emb[_v], _lbl[_v], sample_size=min(2000, int(_v.sum())), random_state=42)
                _d = _db(_emb[_v], _lbl[_v])
                _dbcv = _cl.relative_validity_
            else:
                _s, _d, _dbcv = -1.0, 999.0, -1.0

            _tw_n = min(2000, _emb.shape[0])
            _tw_idx = np.random.RandomState(42).choice(_emb.shape[0], _tw_n, replace=False)
            _trust = _tw(X_weighted[_tw_idx], _emb[_tw_idx], n_neighbors=min(15, _tw_n - 1))

            umap_sweep_results.append({
                'n_components': _nc, 'n_neighbors': _nn, 'metric': _um, 'min_dist': _md,
                'silhouette': round(_s, 3), 'davies_bouldin': round(_d, 3),
                'trustworthiness': round(_trust, 3), 'dbcv': round(_dbcv, 3),
                'n_clusters': _n_c, 'outlier_pct': round(_op, 3)})
            print(f'{_nc:<8} {_nn:<8} {_um:<13} {_md:<7} {_s:<8.3f} {_d:<8.3f} {_trust:<8.3f} {_dbcv:<8.3f} {_n_c:<8} {_op:.1%}')
        except Exception as _e:
            print(f'{_nc:<8} {_nn:<8} {_um:<13} {_md:<7} ERROR: {str(_e)[:50]}')

    umap_sweep_df = pd.DataFrame(umap_sweep_results)
    print(f'\nTop 15 by DBCV:')
    print(umap_sweep_df.sort_values('dbcv', ascending=False).head(15).to_string(index=False))

    print(f'\nPer-metric averages:')
    _ms = umap_sweep_df.groupby('metric').agg(
        avg_sil=('silhouette', 'mean'), best_sil=('silhouette', 'max'),
        avg_trust=('trustworthiness', 'mean'),
        avg_dbcv=('dbcv', 'mean'), best_dbcv=('dbcv', 'max'),
        avg_clusters=('n_clusters', 'mean'), avg_outlier=('outlier_pct', 'mean')
    ).round(3)
    print(_ms.to_string())

    print(f'\nPer-min_dist averages:')
    _mds = umap_sweep_df.groupby('min_dist').agg(
        avg_sil=('silhouette', 'mean'), avg_dbcv=('dbcv', 'mean'),
        avg_trust=('trustworthiness', 'mean'), avg_clusters=('n_clusters', 'mean')
    ).round(3)
    print(_mds.to_string())
else:
    print('UMAP sweep: SKIPPED (RUN_UMAP_SWEEP = False)')

In [ ]:
try:
    umap_sweep_df
except NameError:
    print('No UMAP sweep data — set RUN_UMAP_SWEEP = True')
else:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    _metrics = sorted(umap_sweep_df['metric'].unique())
    _mc = {'correlation': '#E74C3C', 'cosine': '#3498DB', 'euclidean': '#2ECC71'}

    fig = make_subplots(rows=2, cols=3, subplot_titles=(
        'Silhouette by Metric', 'DBCV by Metric', 'Trustworthiness by Metric',
        'DBCV vs n_components (by metric)', 'DBCV vs min_dist (by metric)',
        'Best Config per Metric (DBCV)'))

    for m in _metrics:
        s = umap_sweep_df[umap_sweep_df['metric'] == m]
        c = _mc.get(m, '#888')
        fig.add_trace(go.Histogram(x=s['silhouette'], name=m, marker_color=c, opacity=0.7), row=1, col=1)
        fig.add_trace(go.Histogram(x=s['dbcv'], name=m, marker_color=c, opacity=0.7,
                                   showlegend=False), row=1, col=2)
        fig.add_trace(go.Histogram(x=s['trustworthiness'], name=m, marker_color=c, opacity=0.7,
                                   showlegend=False), row=1, col=3)
        by_nc = s.groupby('n_components').agg(dbcv=('dbcv', 'mean')).reset_index()
        fig.add_trace(go.Scatter(x=by_nc['n_components'], y=by_nc['dbcv'], mode='markers+lines',
                                 name=m, marker=dict(size=8, color=c), showlegend=False), row=2, col=1)
        by_md = s.groupby('min_dist').agg(dbcv=('dbcv', 'mean')).reset_index()
        fig.add_trace(go.Scatter(x=by_md['min_dist'], y=by_md['dbcv'], mode='markers+lines',
                                 name=m, marker=dict(size=8, color=c), showlegend=False), row=2, col=2)

    best = umap_sweep_df.loc[umap_sweep_df.groupby('metric')['dbcv'].idxmax()]
    fig.add_trace(go.Bar(x=best['metric'], y=best['dbcv'],
                         marker_color=[_mc.get(m, '#888') for m in best['metric']],
                         text=[f'nc={r.n_components} nn={r.n_neighbors} md={r.min_dist}'
                               for _, r in best.iterrows()],
                         textposition='auto', showlegend=False), row=2, col=3)

    fig.update_xaxes(title_text='Silhouette', row=1, col=1)
    fig.update_xaxes(title_text='DBCV', row=1, col=2)
    fig.update_xaxes(title_text='Trustworthiness', row=1, col=3)
    fig.update_xaxes(title_text='n_components', row=2, col=1)
    fig.update_yaxes(title_text='Avg DBCV', row=2, col=1)
    fig.update_xaxes(title_text='min_dist', row=2, col=2)
    fig.update_yaxes(title_text='Avg DBCV', row=2, col=2)
    fig.update_yaxes(title_text='Best DBCV', row=2, col=3)
    fig.update_layout(height=700, template='plotly_dark', title_text='UMAP Sweep — Metric + min_dist + Quality')
    fig.show()

## 8 — HDBSCAN

In [ ]:
from sklearn.metrics import silhouette_score

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
    min_samples=HDBSCAN_MIN_SAMPLES,
    cluster_selection_epsilon=HDBSCAN_EPSILON,
    metric='euclidean',
    cluster_selection_method=HDBSCAN_SELECTION_METHOD,
    prediction_data=True,
    gen_min_span_tree=True
)
cluster_labels = clusterer.fit_predict(embedding_cluster)
probabilities = clusterer.probabilities_

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_outliers = int(np.sum(cluster_labels == -1))
n_total = len(cluster_labels)

sil = silhouette_score(
    embedding_cluster[cluster_labels != -1],
    cluster_labels[cluster_labels != -1],
    sample_size=min(2000, int((cluster_labels != -1).sum())),
    random_state=42
) if n_clusters >= 2 else -1

dbcv_score = clusterer.relative_validity_

print(f'HDBSCAN complete')
print(f'  Clusters:   {n_clusters}')
print(f'  Outliers:   {n_outliers} ({100 * n_outliers / n_total:.1f}%)')
print(f'  Silhouette: {sil:.3f}')
print(f'  DBCV:       {dbcv_score:.3f}')

### 8.1 — HDBSCAN Parameter Sweep
Sweeps `min_cluster_size` × `min_samples` × `epsilon` × `selection_method`
on the fixed UMAP embedding. Measures silhouette, DBCV, persistence, avg probability.

In [ ]:
if RUN_HDBSCAN_SWEEP:
    from itertools import product as _product
    from sklearn.metrics import silhouette_score as _sil

    _mcs_list = [5, 10, 15, 20, 30, 40, 60]
    _ms_list  = [1, 3, 5, 10, 15]
    _eps_list = [0.0, 0.1, 0.2, 0.3, 0.5]
    _sel_list = ['eom', 'leaf']

    hdb_sweep_results = []
    _total = len(_mcs_list) * len(_ms_list) * len(_eps_list) * len(_sel_list)
    print(f'HDBSCAN sweep: {_total} combos')
    print(f'{"mcs":<6} {"ms":<5} {"eps":<6} {"sel":<6} {"clust":<8} {"out%":<8} {"sil":<8} {"DBCV":<8} {"avg_prob"}')
    print('-' * 70)

    _all_labels = []

    for _mcs, _ms, _eps, _sel in _product(_mcs_list, _ms_list, _eps_list, _sel_list):
        _cl = hdbscan.HDBSCAN(min_cluster_size=_mcs, min_samples=_ms,
                               cluster_selection_epsilon=_eps, metric='euclidean',
                               cluster_selection_method=_sel,
                               gen_min_span_tree=True, prediction_data=True)
        _lbl = _cl.fit_predict(embedding_cluster)
        _n_c = len(set(_lbl)) - (1 if -1 in _lbl else 0)
        _n_o = int(np.sum(_lbl == -1))
        _op = _n_o / len(_lbl)

        if _n_c >= 2:
            _v = _lbl != -1
            _s = _sil(embedding_cluster[_v], _lbl[_v],
                       sample_size=min(2000, int(_v.sum())), random_state=42)
            _dbcv = _cl.relative_validity_
        else:
            _s, _dbcv = -1.0, -1.0

        _avg_prob = float(_cl.probabilities_[_lbl != -1].mean()) if _n_c >= 1 else 0.0
        _all_labels.append(_lbl.copy())

        hdb_sweep_results.append({
            'min_cluster_size': _mcs, 'min_samples': _ms, 'epsilon': _eps,
            'selection_method': _sel,
            'n_clusters': _n_c, 'outlier_pct': round(_op, 3),
            'silhouette': round(_s, 3), 'dbcv': round(_dbcv, 3),
            'avg_probability': round(_avg_prob, 3)})

    hdb_sweep_df = pd.DataFrame(hdb_sweep_results)

    # Cluster persistence: pairwise ARI between neighbouring runs
    from sklearn.metrics import adjusted_rand_score as _ari
    _n_runs = len(_all_labels)
    _ari_per_run = []
    for i in range(_n_runs):
        _aris = []
        for j in range(max(0, i-5), min(_n_runs, i+6)):
            if i != j:
                _aris.append(_ari(_all_labels[i], _all_labels[j]))
        _ari_per_run.append(round(np.mean(_aris), 3) if _aris else 0.0)
    hdb_sweep_df['persistence'] = _ari_per_run

    print(f'\nTop 10 by DBCV:')
    print(hdb_sweep_df.sort_values('dbcv', ascending=False).head(10)[
        ['min_cluster_size', 'min_samples', 'epsilon', 'selection_method',
         'n_clusters', 'dbcv', 'silhouette', 'persistence', 'avg_probability']
    ].to_string(index=False))

    print(f'\nTop 10 by persistence:')
    print(hdb_sweep_df.sort_values('persistence', ascending=False).head(10)[
        ['min_cluster_size', 'min_samples', 'epsilon', 'selection_method',
         'n_clusters', 'dbcv', 'silhouette', 'persistence', 'avg_probability']
    ].to_string(index=False))

    print(f'\nEOM vs Leaf summary:')
    _sel_summary = hdb_sweep_df.groupby('selection_method').agg(
        avg_sil=('silhouette', 'mean'), best_sil=('silhouette', 'max'),
        avg_dbcv=('dbcv', 'mean'), best_dbcv=('dbcv', 'max'),
        avg_clusters=('n_clusters', 'mean'),
        avg_persistence=('persistence', 'mean'),
        avg_outlier=('outlier_pct', 'mean')
    ).round(3)
    print(_sel_summary.to_string())

    print(f'\nTotal combos: {len(hdb_sweep_df)}')
else:
    print('HDBSCAN sweep: SKIPPED (RUN_HDBSCAN_SWEEP = False)')

In [ ]:
try:
    hdb_sweep_df
except NameError:
    print('No HDBSCAN sweep data — set RUN_HDBSCAN_SWEEP = True')
else:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    _sc = {'eom': '#E74C3C', 'leaf': '#3498DB'}

    fig = make_subplots(rows=2, cols=3, subplot_titles=(
        'DBCV: EOM vs Leaf', 'Silhouette: EOM vs Leaf', 'Persistence: EOM vs Leaf',
        'DBCV vs Silhouette', 'DBCV vs Persistence', 'Clusters vs Avg Probability'))

    for sel in ['eom', 'leaf']:
        s = hdb_sweep_df[hdb_sweep_df['selection_method'] == sel]
        c = _sc[sel]
        fig.add_trace(go.Histogram(x=s['dbcv'], name=sel, marker_color=c, opacity=0.7), row=1, col=1)
        fig.add_trace(go.Histogram(x=s['silhouette'], name=sel, marker_color=c, opacity=0.7,
                                   showlegend=False), row=1, col=2)
        fig.add_trace(go.Histogram(x=s['persistence'], name=sel, marker_color=c, opacity=0.7,
                                   showlegend=False), row=1, col=3)

    fig.add_trace(go.Scatter(
        x=hdb_sweep_df['silhouette'], y=hdb_sweep_df['dbcv'], mode='markers',
        marker=dict(size=5,
                    color=[_sc.get(m, '#888') for m in hdb_sweep_df['selection_method']],
                    opacity=0.6),
        showlegend=False), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=hdb_sweep_df['persistence'], y=hdb_sweep_df['dbcv'], mode='markers',
        marker=dict(size=5, color=hdb_sweep_df['outlier_pct']*100, colorscale='Inferno',
                    showscale=True, colorbar=dict(title='Out%', x=0.68)),
        showlegend=False), row=2, col=2)

    fig.add_trace(go.Scatter(
        x=hdb_sweep_df['n_clusters'], y=hdb_sweep_df['avg_probability'], mode='markers',
        marker=dict(size=5,
                    color=[_sc.get(m, '#888') for m in hdb_sweep_df['selection_method']],
                    opacity=0.6),
        showlegend=False), row=2, col=3)

    fig.update_xaxes(title_text='DBCV', row=1, col=1)
    fig.update_xaxes(title_text='Silhouette', row=1, col=2)
    fig.update_xaxes(title_text='Persistence (ARI)', row=1, col=3)
    fig.update_xaxes(title_text='Silhouette', row=2, col=1)
    fig.update_yaxes(title_text='DBCV', row=2, col=1)
    fig.update_xaxes(title_text='Persistence', row=2, col=2)
    fig.update_yaxes(title_text='DBCV', row=2, col=2)
    fig.update_xaxes(title_text='n_clusters', row=2, col=3)
    fig.update_yaxes(title_text='Avg Probability', row=2, col=3)
    fig.update_layout(height=700, template='plotly_dark', title_text='HDBSCAN Sweep — EOM vs Leaf + Quality')
    fig.show()

## 9 — LLM Cluster Naming (Ollama)

Uses a local Ollama model to name each cluster based on discriminative features,
unique differentiators, and sample movie titles. Requires `ollama serve`.

In [ ]:
import requests
import json as json_module

# ── Centroids for discriminative feature computation ─────────────────────────
centroid_df = pd.DataFrame(X_weighted, index=feature_matrix.index, columns=feature_names)
centroid_df['_cluster'] = cluster_labels
centroid_df = centroid_df.groupby('_cluster').mean()

def get_discriminative_features(cluster_id, top_n=20):
    global_mean  = centroid_df.mean(axis=0)
    delta        = centroid_df.loc[cluster_id] - global_mean
    top_positive = delta.nlargest(top_n)
    return top_positive[top_positive > 0]

def get_sub_discriminative_features(parent_cluster_id, sub_cluster_id, top_n=15):
    parent_mask = cluster_labels == parent_cluster_id
    sub_labels  = sub_clusters[parent_cluster_id]
    sub_mask_within = sub_labels == sub_cluster_id

    parent_indices = np.where(parent_mask)[0]
    sub_indices    = parent_indices[sub_mask_within]

    sub_centroid    = X[sub_indices].mean(axis=0)
    parent_centroid = X[parent_mask].mean(axis=0)

    delta = pd.Series(sub_centroid - parent_centroid, index=feature_names)
    top_positive = delta.nlargest(top_n)
    return top_positive[top_positive > 0]

def build_all_cluster_top_features(top_n=10):
    all_tops = {}
    for cid in range(n_clusters):
        top_pos = get_discriminative_features(cid, top_n=top_n)
        all_tops[cid] = list(top_pos.index)
    return all_tops

def get_differentiating_features(cluster_id, all_tops, top_n=10, extra_n=5):
    top_pos = get_discriminative_features(cluster_id, top_n=top_n)
    my_features = list(top_pos.index)

    shared = set()
    for other_id, other_feats in all_tops.items():
        if other_id == cluster_id:
            continue
        shared |= set(my_features) & set(other_feats)

    extra_unique = []
    if shared:
        extended = get_discriminative_features(cluster_id, top_n=top_n + 20)
        for feat in extended.index:
            if feat not in my_features:
                is_unique = True
                for other_id, other_feats in all_tops.items():
                    if other_id == cluster_id:
                        continue
                    if feat in other_feats:
                        is_unique = False
                        break
                if is_unique:
                    extra_unique.append((feat, float(extended[feat])))
                    if len(extra_unique) >= extra_n:
                        break

    return top_pos, list(shared), extra_unique

def format_features_short(top_pos, n=5):
    return [f'{f} (+{v:.2f})' for f, v in list(top_pos.items())[:n]]

def query_ollama(prompt, model=None):
    model = model or OLLAMA_MODEL
    try:
        response = requests.post(
            f'{OLLAMA_BASE_URL}/api/generate',
            json={'model': model, 'prompt': prompt, 'stream': False,
                  'options': {'temperature': 0.3}},
            timeout=180
        )
        data = response.json()
        if 'error' in data:
            raise RuntimeError(f"Ollama error: {data['error']}")
        return data['response'].strip()
    except Exception as e:
        print(f'  ⚠ Ollama call failed: {e}')
        return f'{{"name": "Unnamed", "description": "LLM unavailable: {e}"}}'

def name_cluster(cluster_id, all_tops, sample_tt_codes=None, title_lookup=None):
    top_pos, shared, extra_unique = get_differentiating_features(cluster_id, all_tops)
    pos_str = '\n'.join([f'  - {f}: +{v:.3f} above average' for f, v in top_pos.items()])

    diff_str = ''
    if shared and extra_unique:
        shared_list = ', '.join(sorted(shared))
        extra_list = '\n'.join([f'  - {f}: +{v:.3f} above average (unique to this cluster)'
                                for f, v in extra_unique])
        diff_str = f'''

NOTE: Some top features above (specifically: {shared_list}) also appear as top features
in other clusters. The following features are UNIQUE to this cluster and should help
distinguish its identity:
{extra_list}'''

    sample_str = ''
    if sample_tt_codes and title_lookup:
        titles = [title_lookup.get(str(tt), str(tt)) for tt in sample_tt_codes[:6]]
        sample_str = f'\nExample movies: {", ".join(titles)}'

    prompt = f"""You are a film taxonomy expert. A cluster of movies has been grouped by
the similarity of their content profile. These movies share the same pattern of which
content elements are most prominent.

TOP FEATURES ELEVATED ABOVE AVERAGE IN THIS CLUSTER:
{pos_str}{diff_str}
{sample_str}

Based on this profile, give this cluster a precise, evocative name (3-6 words)
and a one-sentence description of what unifies these films.

Respond in JSON only:
{{"name": "cluster name", "description": "one sentence"}}"""

    raw = query_ollama(prompt)
    try:
        result = json_module.loads(raw)
    except json_module.JSONDecodeError:
        result = {'name': f'Cluster {cluster_id}', 'description': raw}

    result['top_features'] = format_features_short(top_pos)
    if shared:
        result['shared_with_other_rails'] = list(shared)
    if extra_unique:
        result['unique_differentiators'] = [f'{f} (+{v:.2f})' for f, v in extra_unique]
    return result

def name_sub_cluster(parent_cluster_id, sub_cluster_id, parent_name,
                     sibling_tops=None, sample_tt_codes=None, title_lookup=None):
    top_pos = get_sub_discriminative_features(parent_cluster_id, sub_cluster_id)
    my_features = list(top_pos.index)

    shared = set()
    extra_unique = []
    if sibling_tops:
        for other_sid, other_feats in sibling_tops.items():
            if other_sid == sub_cluster_id:
                continue
            shared |= set(my_features[:10]) & set(other_feats[:10])

        if shared:
            extended = get_sub_discriminative_features(parent_cluster_id, sub_cluster_id, top_n=30)
            for feat in extended.index:
                if feat not in my_features[:10]:
                    is_unique = True
                    for other_sid, other_feats in sibling_tops.items():
                        if other_sid == sub_cluster_id:
                            continue
                        if feat in other_feats[:10]:
                            is_unique = False
                            break
                    if is_unique:
                        extra_unique.append((feat, float(extended[feat])))
                        if len(extra_unique) >= 5:
                            break

    pos_str = '\n'.join([f'  - {f}: +{v:.3f} above parent avg' for f, v in top_pos.items()])

    diff_str = ''
    if shared and extra_unique:
        shared_list = ', '.join(sorted(shared))
        extra_list = '\n'.join([f'  - {f}: +{v:.3f} (unique to this sub-rail)'
                                for f, v in extra_unique])
        diff_str = f'''

NOTE: Features shared with sibling sub-rails: {shared_list}
Features UNIQUE to this sub-rail that help distinguish it:
{extra_list}'''

    sample_str = ''
    if sample_tt_codes and title_lookup:
        titles = [title_lookup.get(str(tt), str(tt)) for tt in sample_tt_codes[:6]]
        sample_str = f'\nExample movies: {", ".join(titles)}'

    prompt = f"""You are a film taxonomy expert. This is a SUB-CLUSTER within the larger
rail called "{parent_name}". These movies share the parent rail's broad profile but
have specific features that are more prominent than the parent average.

FEATURES ELEVATED ABOVE THE PARENT RAIL AVERAGE:
{pos_str}{diff_str}
{sample_str}

Give this sub-cluster a precise, evocative name (2-5 words) that describes what
distinguishes it FROM the parent rail "{parent_name}". Also give a one-sentence
description of what makes this sub-group unique within the parent.

Respond in JSON only:
{{"name": "sub-cluster name", "description": "one sentence"}}"""

    raw = query_ollama(prompt)
    try:
        result = json_module.loads(raw)
    except json_module.JSONDecodeError:
        result = {'name': f'Sub {sub_cluster_id}', 'description': raw}

    result['top_features'] = format_features_short(top_pos)
    if shared:
        result['shared_with_siblings'] = list(shared)
    if extra_unique:
        result['unique_differentiators'] = [f'{f} (+{v:.2f})' for f, v in extra_unique]
    return result

print('LLM naming functions defined')
print(f'  Model: {OLLAMA_MODEL}')
print(f'  Endpoint: {OLLAMA_BASE_URL}')

In [ ]:
# Requires: ollama serve  (with the model already pulled)

cluster_names = {}
sub_cluster_names = {}

# ── Pre-compute top features for all clusters (for dedup) ────────────────────
all_cluster_tops = build_all_cluster_top_features(top_n=10)

# ── Name top-level clusters ──────────────────────────────────────────────────
print(f'Naming {n_clusters} clusters via Ollama ({OLLAMA_MODEL})...\n')

for cluster_id in range(n_clusters):
    mask       = cluster_labels == cluster_id
    sample_tts = np.array(tt_codes)[mask][:6].tolist()
    result     = name_cluster(cluster_id, all_cluster_tops, sample_tts, title_lookup)
    cluster_names[cluster_id] = result

    avg_conf = probabilities[mask].mean()
    print(f'{"="*70}')
    print(f'Rail {cluster_id}: {result["name"]}  ({mask.sum()} films, {avg_conf:.0%} avg conf)')
    print(f'  {result["description"]}')
    print(f'  Top features: {", ".join(result["top_features"])}')
    if result.get('shared_with_other_rails'):
        print(f'  Shared with other rails: {", ".join(result["shared_with_other_rails"])}')
    if result.get('unique_differentiators'):
        print(f'  Unique differentiators: {", ".join(result["unique_differentiators"])}')

# ── Name sub-clusters ────────────────────────────────────────────────────────
print(f'\n\n{"="*70}')
print(f'Naming sub-clusters via Ollama...')
print(f'{"="*70}\n')

for parent_id in sorted(sub_clusters.keys()):
    sub_labels = sub_clusters[parent_id]
    unique_subs = sorted(set(sub_labels))
    unique_subs = [s for s in unique_subs if s != -1]

    if not unique_subs:
        continue

    parent_name = cluster_names.get(parent_id, {}).get('name', f'Cluster {parent_id}')
    parent_mask = cluster_labels == parent_id
    parent_indices = np.where(parent_mask)[0]

    # Pre-compute sibling tops for dedup
    sibling_tops = {}
    for sid in unique_subs:
        sub_pos = get_sub_discriminative_features(parent_id, sid, top_n=10)
        sibling_tops[sid] = list(sub_pos.index)

    print(f'\nRail {parent_id}: {parent_name} ({len(unique_subs)} sub-rails)')
    print(f'  {"-"*65}')

    for sub_id in unique_subs:
        sub_mask_within = sub_labels == sub_id
        sub_indices = parent_indices[sub_mask_within]
        sub_size = len(sub_indices)

        sub_tts = np.array(tt_codes)[sub_indices][:6].tolist()
        avg_sub_conf = probabilities[sub_indices].mean()

        result = name_sub_cluster(parent_id, sub_id, parent_name,
                                  sibling_tops, sub_tts, title_lookup)
        sub_cluster_names[(parent_id, sub_id)] = result

        print(f'  Sub {sub_id}: {result["name"]}  ({sub_size} films, {avg_sub_conf:.0%} avg conf)')
        print(f'    {result["description"]}')
        print(f'    Top features (vs parent): {", ".join(result["top_features"])}')
        if result.get('unique_differentiators'):
            print(f'    Unique differentiators: {", ".join(result["unique_differentiators"])}')

    n_sub_out = int(np.sum(sub_labels == -1))
    if n_sub_out > 0:
        print(f'  Sub-outliers: {n_sub_out} films')

print(f'\nNaming complete: {len(cluster_names)} rails + {len(sub_cluster_names)} sub-rails')

## 10 — Export Artifacts

In [ ]:
import joblib
from datetime import datetime

results_dir = Path(RESULTS_DIR)
results_dir.mkdir(exist_ok=True)

artifacts = {
    'embedding_cluster': embedding_cluster,
    'embedding_3d': embedding_3d,
    'embedding_2d': embedding_2d,
    'cluster_labels': cluster_labels,
    'probabilities': clusterer.probabilities_,
    'outlier_scores': clusterer.outlier_scores_,
    'n_clusters': n_clusters,
    'sub_clusters': {},
    'sub_embeddings_2d': {},
    'X': X,
    'X_weighted': X_weighted,
    'feature_names': feature_names,
    'tt_codes': tt_codes,
    'laplacian_scores': lap_scores,
    'laplacian_weights': lap_weights,
    'cluster_names': cluster_names,
    'sub_cluster_names': {f'{k[0]}_{k[1]}': v for k, v in sub_cluster_names.items()},
    'title_lookup': title_lookup,
    'jordan_df': jordan_df,
    'config': {
        'PIPELINE': 'graph_theory',
        'LAPLACIAN_K_NEIGHBORS': LAPLACIAN_K_NEIGHBORS,
        'LAPLACIAN_METRIC': LAPLACIAN_METRIC,
        'UMAP_N_COMPONENTS': UMAP_N_COMPONENTS,
        'UMAP_N_NEIGHBORS': UMAP_N_NEIGHBORS,
        'UMAP_MIN_DIST': UMAP_MIN_DIST,
        'UMAP_METRIC': UMAP_METRIC,
        'HDBSCAN_MIN_CLUSTER_SIZE': HDBSCAN_MIN_CLUSTER_SIZE,
        'HDBSCAN_MIN_SAMPLES': HDBSCAN_MIN_SAMPLES,
        'HDBSCAN_EPSILON': HDBSCAN_EPSILON,
        'HDBSCAN_SELECTION_METHOD': HDBSCAN_SELECTION_METHOD,
        'INCLUDE_GENRE_FEATURES': INCLUDE_GENRE_FEATURES,
        'INCLUDE_DECADE_FEATURE': INCLUDE_DECADE_FEATURE,
        'OLLAMA_MODEL': OLLAMA_MODEL,
        'ROBUSTSCALER': False,
        'L2_NORMALIZE': False,
        'timestamp': datetime.now().isoformat(),
    }
}

out_path = results_dir / 'pipeline_artifacts.pkl'
joblib.dump(artifacts, out_path, compress=('zlib', 3))
print(f'Pipeline artifacts saved to {out_path}')
print(f'Keys: {list(artifacts.keys())}')

## Summary

In [ ]:
print('=' * 60)
print('PIPELINE SUMMARY — GRAPH THEORY')
print('=' * 60)
print(f'  Movies:          {X.shape[0]:,}')
print(f'  Features:        {X.shape[1]} (genome + genre + decade)')
print(f'  Preprocessing:   Clean only (no scaling)')
print(f'  Laplacian:       k={LAPLACIAN_K_NEIGHBORS}, metric={LAPLACIAN_METRIC}')
print(f'  UMAP:            {UMAP_N_COMPONENTS}D, {UMAP_N_NEIGHBORS} neighbors, min_dist={UMAP_MIN_DIST}, {UMAP_METRIC}')
print(f'  HDBSCAN:         min_size={HDBSCAN_MIN_CLUSTER_SIZE}, min_samples={HDBSCAN_MIN_SAMPLES}, eps={HDBSCAN_EPSILON}, {HDBSCAN_SELECTION_METHOD}')
print(f'  Clusters:        {n_clusters}')
print(f'  Outliers:        {n_outliers} ({100 * n_outliers / n_total:.1f}%)')
print(f'  Silhouette:      {sil:.3f}')
print(f'  DBCV:            {dbcv_score:.3f}')
print(f'  Cluster names:   {len(cluster_names)}')
print(f'  Ollama model:    {OLLAMA_MODEL}')
print('=' * 60)

## 12 — Launch Dashboard

Starts the Streamlit dashboard pointing at the freshly exported artifacts.
The dashboard opens in a new browser tab. Stop the cell to kill the server.

In [ ]:
import subprocess, webbrowser, time

# Launch Streamlit in the background
proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--browser.gatherUsageStats', 'false'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

time.sleep(3)
webbrowser.open('http://localhost:8501')
print(f'Dashboard launched at http://localhost:8501')
print(f'Process PID: {proc.pid}')
print(f'Stop this cell or run proc.terminate() to shut down.')